# §2.3.4 — 교차엔트로피를 어떻게 계산하면 안 되는가

> 딥러닝 교재 · 1부 2장 3절 4항 (🐍)
> 선행: §2.3.1(유도와 로짓 형태) · §2.3.2(기울기 $q - \mathbf{y}$)

## 이 노트북이 답하는 질문

1. **`log(softmax(z))` 는 언제 깨지는가?** float32에서 로짓 크기를 키우며 확인한다.
2. **소프트맥스를 이동으로 안정화하면 충분한가?** — **충분하지 않다.** 이것이 이 노트북의 핵심이다.
3. **float16에서는 언제 깨지는가?** 답은 로짓 크기 **11** 근처다. 혼합 정밀도 학습에서 실제로 만나는 값이다.
4. **손실만 깨지는가, 기울기도 깨지는가?**

**예상 실행 시간** CPU 단일 코어 약 10초.

이 노트북은 성능 최적화에 대한 것이 아니다. **정답이 나오느냐 `nan` 이 나오느냐의 문제**다.

---
## 0. 설정

In [ ]:
import os, glob, time, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')   # 아래넘침·넘침 경고를 직접 관찰하므로 끈다

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
SEED     = 20260807
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_2_3_4_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 부동소수점의 벽

$\exp$ 은 정의역을 조금만 벗어나도 표현 범위를 넘는다. 임계값을 먼저 확인한다.

$$\exp(z) \ \text{넘침} \iff z > \log(\text{max}), \qquad \exp(z) \ \text{아래넘침} \iff z < \log(\text{tiny})$$

In [ ]:
print("                max          tiny        exp 넘침 z    exp 아래넘침 z    eps")
for dt, nm in [(np.float16,'float16'), (np.float32,'float32'), (np.float64,'float64')]:
    fi = np.finfo(dt)
    print(f"  {nm:>8}   {float(fi.max):.3e}  {float(fi.tiny):.3e}    "
          f"{np.log(float(fi.max)):>8.2f}      {np.log(float(fi.tiny)):>8.2f}   {float(fi.eps):.1e}")
print("\n-> float32는 로짓이 89를 넘으면 exp가 무한대가 된다.")
print("   float16은 11을 넘으면 그렇게 된다. 로짓 11은 전혀 극단적인 값이 아니다.")

---
## 2. 세 가지 구현

**(A) 순진한 구현.** 정의를 그대로 옮긴 것.

$$q_k = \frac{e^{z_k}}{\sum_j e^{z_j}}, \qquad \ell = -\log q_c$$

**(B) 이동한 소프트맥스 + 로그.** 넘침을 막으려고 최댓값을 빼 준다.

$$m = \max_j z_j, \qquad q_k = \frac{e^{z_k - m}}{\sum_j e^{z_j - m}}, \qquad \ell = -\log q_c$$

**(C) 로그 소프트맥스.** 로그를 안으로 밀어 넣어 지수를 한 번만 쓴다.

$$\log q_k = (z_k - m) - \log\sum_j e^{z_j - m}$$

### 왜 최댓값을 빼도 되는가

소프트맥스는 로짓 전체의 평행이동에 불변이다. 임의의 상수 $c$ 에 대해

$$\frac{e^{z_k - c}}{\sum_j e^{z_j - c}} = \frac{e^{-c}e^{z_k}}{e^{-c}\sum_j e^{z_j}} = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

**분자와 분모에서 $e^{-c}$ 가 상쇄된다.** 수학적으로는 아무것도 바뀌지 않고, **표현 범위 안으로 들어오는 것만 바뀐다.**
§2.4.1이 이 불변성을 정식으로 다루며, §2.3.2에서 기울기 성분의 합이 0이었던 것이 같은 사실의 다른 면이다.

In [ ]:
def impl_naive(z):
    e = np.exp(z)
    return np.log(e / e.sum(axis=-1, keepdims=True))

def impl_shift_then_log(z):
    m = z.max(axis=-1, keepdims=True)
    e = np.exp(z - m)
    return np.log(e / e.sum(axis=-1, keepdims=True))

def impl_log_softmax(z):
    m = z.max(axis=-1, keepdims=True)
    s = z - m
    return s - np.log(np.exp(s).sum(axis=-1, keepdims=True))

IMPLS = [('A ' + lab('순진','naive'), impl_naive, CB[4]),
         ('B ' + lab('이동+로그','shift+log'), impl_shift_then_log, CB[1]),
         ('C ' + lab('로그 소프트맥스','log_softmax'), impl_log_softmax, CB[3])]

# 평행이동 불변성 수치 확인 (안전한 범위에서)
z0 = np.array([2.0, 1.0, 0.0, -1.0])
for c in [0.0, 10.0, -10.0]:
    print(f"  c={c:+6.1f}: -log q_1 = {-impl_log_softmax(z0 + c)[0]:.12f}")
print("-> 평행이동해도 값이 같다 (§2.4.1)")

---
## 3. 실험 A — 로짓 전체가 커질 때 (넘침)

로짓의 **패턴은 그대로 두고 전체를 평행이동**한다. 2절에 의해 참값은 변하지 않아야 한다.

In [ ]:
PAT = np.array([2.0, 1.0, 0.0, -1.0])
REF = float(-impl_log_softmax(PAT)[0])       # float64 참값
CS = np.array([0, 20, 50, 80, 86, 88, 90, 100, 300, 1000], dtype=float)

print(f"참값 -log q_1 = {REF:.6f}  (모든 c에서 같아야 한다)\n")
print("     c        A 순진      B 이동+로그   C 로그softmax")
tabA = {nm: [] for nm, _, _ in IMPLS}
for c in CS:
    z = (PAT + c).astype(np.float32)
    row = []
    for nm, f, _ in IMPLS:
        v = float(-f(z)[0]); tabA[nm].append(v); row.append(v)
    print(f"  {c:>6.0f}   {row[0]:>10.5f}   {row[1]:>10.5f}   {row[2]:>12.5f}")
print(f"\n-> A는 c = {np.log(np.finfo(np.float32).max):.1f} 를 넘는 순간 nan 이 된다.")
print("   exp(z)가 inf가 되고 inf/inf = nan 이기 때문이다.")

---
## 4. 실험 B — 확신이 커질 때 (아래넘침)

**여기가 핵심이다.** 이번에는 로짓의 크기가 아니라 **격차**를 키운다.

$$z = (0,\ -g)$$

정답이 두 번째 클래스일 때의 손실은 $\ell = g + \log(1 + e^{-g}) \approx g$ 로 **$g$ 에 선형으로 자란다.**
모형이 매우 확신하며 틀린 상황이고, 학습 중에 실제로 일어난다.

**A와 B 모두 로짓이 크지 않으므로 넘침은 없다.** 그런데도 깨진다.

In [ ]:
GS = np.array([10, 50, 80, 87, 90, 95, 100, 110, 120, 200], dtype=float)
print("   격차 g      A 순진      B 이동+로그   C 로그softmax      참값")
tabB = {nm: [] for nm, _, _ in IMPLS}
refB = []
for g in GS:
    z = np.array([0.0, -g], dtype=np.float32)
    true = float(g + np.log1p(np.exp(-np.float64(g))))
    refB.append(true)
    row = []
    for nm, f, _ in IMPLS:
        v = float(-f(z)[1]); tabB[nm].append(v); row.append(v)
    print(f"  {g:>7.0f}   {row[0]:>10.4f}   {row[1]:>10.4f}   {row[2]:>12.4f}   {true:>10.4f}")
print("\n-> B(이동)는 넘침을 막았지만 아래넘침은 못 막는다.")
print("   e^{-g} 가 0으로 뭉개지면 소프트맥스가 0을 내고 log(0) = -inf 가 된다.")
print("   C는 로그를 안으로 밀어 넣어 지수를 거치지 않으므로 정확하다.")

> ### 이동만으로는 충분하지 않다
>
> 소프트맥스를 이동으로 안정화하는 것은 널리 알려져 있고, 그것으로 끝이라고 생각하기 쉽다.
> **그러나 이동이 막는 것은 넘침뿐이다.**
>
> $$\text{넘침}: \ \max_j z_j \ \text{가 클 때} \qquad \longrightarrow \ \text{이동으로 해결}$$
> $$\text{아래넘침}: \ z_c - \max_j z_j \ \text{가 매우 음수일 때} \qquad \longrightarrow \ \textbf{이동으로 해결 안 됨}$$
>
> 둘째는 **모형이 확신하며 틀렸을 때** 일어난다. 그리고 그때가 바로 손실이 커야 하는 순간이다.
> **가장 중요한 표본에서 값을 잃는 것**이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8))

for nm, f, c in IMPLS:
    v = np.array(tabA[nm])
    err = np.abs(v - REF)/abs(REF)
    err = np.where(np.isfinite(err), err, 1e2)          # nan/inf 는 위쪽에 표시
    axes[0].semilogy(CS, np.maximum(err, 1e-9), 'o-', ms=4, color=c, label=nm)
axes[0].axvline(np.log(np.finfo(np.float32).max), color=CB[0], ls='--', lw=1.2,
                label=lab(r'$\log(\mathrm{max}_{32}) = 88.7$', r'$\log(\mathrm{max}_{32})$'))
axes[0].axhline(1e2, color='0.7', lw=0.8)
axes[0].text(2, 1.5e2, lab('nan / inf', 'nan / inf'), fontsize=8, color='0.4')
axes[0].set_xlabel(lab('로짓에 더한 상수 $c$', 'constant added to logits'))
axes[0].set_ylabel(lab('상대 오차', 'relative error'))
axes[0].set_title(lab('A. 넘침 — 로짓이 클 때', 'A. overflow'), fontsize=10)
axes[0].legend(fontsize=7.5)

for nm, f, c in IMPLS:
    v = np.array(tabB[nm]); r = np.array(refB)
    err = np.abs(v - r)/r
    err = np.where(np.isfinite(err), err, 1e2)
    axes[1].semilogy(GS, np.maximum(err, 1e-9), 'o-', ms=4, color=c, label=nm)
axes[1].axvline(-np.log(np.finfo(np.float32).tiny), color=CB[0], ls='--', lw=1.2,
                label=lab(r'$-\log(\mathrm{tiny}_{32}) = 87.3$', r'$-\log(\mathrm{tiny}_{32})$'))
axes[1].axhline(1e2, color='0.7', lw=0.8)
axes[1].set_xlabel(lab('로짓 격차 $g$', 'logit gap $g$'))
axes[1].set_title(lab('B. 아래넘침 — 확신하며 틀릴 때', 'B. underflow'), fontsize=10)
axes[1].legend(fontsize=7.5)
fig.suptitle(lab('이동(B)은 왼쪽만 고친다 — 오른쪽에서는 순진한 구현과 똑같이 무너진다',
                 'shifting fixes only the left panel'), y=1.04, fontsize=10)
show('overflow_underflow')

---
## 5. 기울기도 함께 깨진다

§2.3.2에서 $\nabla_z \ell = q - \mathbf{y}$ 였다. $q$ 를 어떻게 얻느냐에 따라 기울기도 달라진다.

In [ ]:
def q_naive(z):
    e = np.exp(z); return e/e.sum(axis=-1, keepdims=True)
def q_stable(z):
    return np.exp(impl_log_softmax(z))

print("정답이 클래스 0일 때의 기울기 (q - y)")
print("     상황            q_naive 로 계산        q_stable 로 계산")
cases = [(lab('평범','normal'), np.array([2.,1.,0.,-1.], dtype=np.float32)),
         (lab('로짓 큼 c=100','large logits'), np.array([102.,101.,100.,99.], dtype=np.float32)),
         (lab('확신하며 틀림 g=120','confidently wrong'), np.array([-120., 0.], dtype=np.float32))]
for nm, z in cases:
    y = np.zeros_like(z); y[0] = 1.0
    gn = q_naive(z) - y; gs = q_stable(z) - y
    print(f"  {nm:>18}   {np.array2string(gn, precision=4, floatmode='fixed'):>26}   "
          f"{np.array2string(gs, precision=4, floatmode='fixed')}")
print("\n[읽을 것 — 두 실패가 성격이 다르다]")
print("  넘침(로짓 큼): 손실도 기울기도 nan. 한 번 나오면 파라미터 전체가 오염된다.")
print("  아래넘침(확신하며 틀림): 손실은 inf 인데 **기울기는 정확하다**.")
print("    q -> 0 이 올바른 극한이므로 q - y = -1 은 맞는 값이고, 이는 가능한 최대 크기다.")
print("    곧 이 경우 학습 자체는 진행되지만 **손실 값을 잃는다**.")
print("    손실 값에 의존하는 모든 것이 무너진다 — 배치 평균, 표본 가중,")
print("    조기 종료 판정, 혼합 정밀도의 손실 스케일링(§53.1).")

> ### 두 실패의 성격이 다르다
>
> | | 손실 | 기울기 | 무엇을 잃는가 |
> |---|---|---|---|
> | **넘침** (로짓이 큼) | `nan` | `nan` | **전부.** 파라미터가 오염되고 복구 불가 |
> | **아래넘침** (확신하며 틀림) | `inf` | **정확** | **손실 값만.** 학습은 진행된다 |
>
> 둘째가 더 교활하다. **모형은 계속 학습되므로 겉보기에는 아무 문제가 없고**, 손실 곡선만
> `inf` 로 뭉개진다. 배치 평균이 `inf` 가 되고, 표본 가중·조기 종료·손실 스케일링이 전부 무의미해진다.
>
> §2.3.2에서 기울기가 $q - \mathbf{y}$ 로 유계였던 것이 여기서 값을 한다 —
> **기울기는 애초에 폭발할 수 없는 양**이었다.

---
## 6. 이진의 경우 — softplus

이진 교차엔트로피도 같은 문제를 갖는다. 로짓 $z$ 와 레이블 $y \in \{0,1\}$ 에 대해

$$\ell = -\big[y\log\sigma(z) + (1-y)\log(1-\sigma(z))\big] = \log\big(1+e^{-z}\big) + (1-y)\,z$$

$\log(1+e^{x})$ 를 **softplus** 라 한다. 이것도 $x$ 가 크면 $e^x$ 가 넘친다. 안정 형태는

$$\mathrm{softplus}(x) = \max(x, 0) + \log\big(1 + e^{-|x|}\big)$$

$|x|$ 에 음수를 붙였으므로 지수의 인자가 언제나 $\le 0$ 이고, $\log(1+\cdot)$ 는 `log1p` 로 계산한다.

In [ ]:
def softplus_naive(x):
    return np.log(1.0 + np.exp(x))
def softplus_stable(x):
    ax = np.abs(x)
    return np.maximum(x, 0) + np.log1p(np.exp(-ax))

print("       x        naive          stable          참값(float64)")
for x in [-100.0, -40.0, -1.0, 0.0, 1.0, 40.0, 88.0, 90.0, 200.0]:
    xf = np.float32(x)
    n_ = float(softplus_naive(xf)); s_ = float(softplus_stable(xf))
    t_ = float(np.maximum(x,0) + np.log1p(np.exp(-abs(np.float64(x)))))
    print(f"  {x:>8.1f}   {n_:>12.6f}   {s_:>12.6f}   {t_:>14.6f}")
print("\n-> x < -40 에서 naive 는 exp(x)가 아래넘침해 정확히 0을 내놓는다.")
print("   참값은 e^x 로 아주 작지만 0이 아니며, log1p 가 그 작은 값을 살린다.")
print("   x > 88 에서는 naive 가 inf 가 된다.")

---
## 7. float16과 혼합 정밀도

1절의 표에서 float16의 넘침 임계가 **11.09** 였다. 로짓 11은 확신도 $\sigma(11) \approx 0.99998$ 정도로
학습 중에 흔히 나오는 값이다.

In [ ]:
print("float16에서 로짓 전체를 c 만큼 이동 (참값 = %.4f)" % REF)
print("      c      A 순진      C 로그softmax")
for c in [0, 5, 9, 10, 11, 12, 20, 50]:
    z16 = (PAT + c).astype(np.float16)
    a = float(-impl_naive(z16)[0]); cc = float(-impl_log_softmax(z16)[0])
    print(f"  {c:>5}   {a:>10.4f}   {cc:>14.4f}")
print("\n-> float16에서는 로짓 10 근처부터 순진한 구현이 무너진다.")
print("   혼합 정밀도 학습에서 손실 계산만은 float32로 올리는 관행이 이 때문이다 (§53.1).")
print("   다만 C조차 float16의 정밀도 한계(eps ~ 1e-3)는 넘지 못한다 — 값의 유효자릿수를 보라.")

---
## 8. 자기 점검

1. 4절에서 B(이동+로그)가 A(순진)와 **같은 지점에서** 깨졌다. 두 구현이 하는 일이 다른데 왜 같은 곳에서 실패하는가?
2. 5절에서 확신하며 틀린 표본의 기울기가 `q_naive` 로는 0처럼 보였다. **학습에 어떤 영향**을 주겠는가?
3. 6절의 `softplus_stable` 에서 $\max(x,0)$ 을 빼고 $\log(1+e^{-|x|})$ 만 쓰면 어떻게 되는가? $x = 5$ 에서 확인하라.
4. 세 구현 모두 float64에서는 문제가 없어 보인다. **그러면 float64를 쓰면 되는가?** (§53.1의 관점에서)
5. 5절에서 아래넘침일 때 기울기가 정확했다. **그렇다면 안정 구현이 왜 필요한가?** 답을 두 가지 이상 대라.

In [ ]:
# 자기 점검 3의 확인
print("softplus 에서 max(x,0) 항을 빠뜨리면")
print("      x     올바른 식      max 항 누락")
for x in [-5.0, 0.0, 5.0, 20.0]:
    ok = float(np.maximum(x,0) + np.log1p(np.exp(-abs(x))))
    bad = float(np.log1p(np.exp(-abs(x))))
    print(f"  {x:>6.1f}   {ok:>10.5f}   {bad:>12.5f}")
print("\n-> 양수 x에서 완전히 틀린다. max(x,0)는 안정화 장치가 아니라 식의 일부다.")
print("   softplus(x) = x + softplus(-x) 라는 항등식에서 나온 것이다.")

---
## 9. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `PAT` | 3절 | (2,1,0,−1) | 로짓 패턴. 격차를 키우면 4절의 문제가 더 일찍 나타난다 |
| `CS` | 3절 | 0 ~ 1000 | 평행이동 격자 |
| `GS` | 4절 | 10 ~ 200 | 격차 격자 |
| dtype | 3·7절 | float32 | `np.float16` 으로 바꾸면 모든 임계가 8배 가까이 내려온다 |

**권하는 첫 실험** — 3절의 `PAT` 을 `np.array([20.0, 0.0, -20.0, -40.0])` 으로 바꾸십시오.
평행이동을 하나도 하지 않아도(**$c = 0$**) 4절의 아래넘침 문제가 나타납니다.
**로짓의 절대 크기가 아니라 격차가 문제**라는 것을 한 줄로 확인하는 방법입니다.

---

## 결론 — 규약으로 남길 것

> **소프트맥스를 계산한 뒤 로그를 취하지 말 것.** 처음부터 `log_softmax` 를 쓰고,
> 이진에서는 `binary_cross_entropy_with_logits` 계열을 쓴다.
>
> 프레임워크가 `cross_entropy(logits, labels)` 를 별도로 제공하는 이유가 이것이며,
> **확률을 먼저 만들어 넘기는 API를 쓰면 그 안정성을 잃는다.**

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")